# Prepare stratified shopping-review splits

This privacy-cleaned notebook combines the platform CSV files, validates their schema, and creates reproducible 80/10/10 train, development, and test splits.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'data' else Path.cwd().resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
OUTPUT_DIR = PROJECT_ROOT / 'new_data'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DTYPES = {
    'reviewId': 'string',
    'content': 'string',
    'score': 'Int64',
    'thumbsUpCount': 'Int64',
    'at': 'Int64',
    'replyContent': 'string',
    'repliedAt': 'Float64',
    'appName': 'string',
}
COLUMNS = list(DTYPES)

In [ ]:
csv_files = sorted(RAW_DIR.glob('*.csv'))
if not csv_files:
    raise FileNotFoundError(f'No CSV files found in {RAW_DIR}')

frames = []
for csv_path in csv_files:
    frame = pd.read_csv(csv_path, dtype=DTYPES, keep_default_na=True)
    missing = set(COLUMNS) - set(frame.columns)
    extra = set(frame.columns) - set(COLUMNS)
    if missing or extra:
        raise ValueError(f'{csv_path.name}: missing={missing}, extra={extra}')
    frames.append(frame[COLUMNS])

data = pd.concat(frames, ignore_index=True)
print(f'Loaded {len(data):,} reviews from {len(csv_files)} files.')

In [ ]:
train_full, test = train_test_split(
    data, test_size=0.10, stratify=data['appName'], random_state=42
)
train, dev = train_test_split(
    train_full, test_size=1 / 9, stratify=train_full['appName'], random_state=42
)
print({'train': train.shape, 'dev': dev.shape, 'test': test.shape})

In [ ]:
train_ids = set(train['reviewId'])
dev_ids = set(dev['reviewId'])
test_ids = set(test['reviewId'])
assert train_ids.isdisjoint(dev_ids)
assert train_ids.isdisjoint(test_ids)
assert dev_ids.isdisjoint(test_ids)

for split_name, frame in {'train': train, 'dev': dev, 'test': test}.items():
    print(f'\n{split_name}:')
    print(frame['appName'].value_counts(normalize=True).sort_index())

In [ ]:
for split_name, frame in {'train': train, 'dev': dev, 'test': test}.items():
    output_path = OUTPUT_DIR / f'{split_name}.csv'
    frame.to_csv(output_path, index=False)
    print(f'Wrote {len(frame):,} rows to {output_path.relative_to(PROJECT_ROOT)}')